# Supplementary: Architecture defaults

Each `ModelArchitectureStrategy` ships with a *default* player partition and masking primitive that matches how the model was trained. You can override either when constructing the architecture or the explainer.

Main tutorials: `tutorial_01_player_masker_strategies.ipynb` (matrix of choices), `tutorial_03_benchmarks_and_comparison.ipynb` (performance).

## Setup

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
from PIL import Image
from torchvision import transforms

import shapiq.vision as _vision_module

# Bundled sample image (same as tutorial_03) — no network access required
DOG_PATH = Path(_vision_module.__file__).parent / "dog.png"
assert DOG_PATH.exists(), f"dog.png not found at {DOG_PATH}"

pil_224 = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])(
    Image.open(DOG_PATH).convert("RGB")
)
image = np.asarray(pil_224)
print("image shape:", image.shape)

image shape: (224, 224, 3)


## Default player and masking per architecture

In [2]:
from types import SimpleNamespace

from shapiq.vision import (
    CLIPArchitecture,
    ConvNeXtArchitecture,
    DINOv2Architecture,
    ResNetArchitecture,
    ViTArchitecture,
)


def describe(arch, label):
    player = arch.default_player_strategy()
    masking = arch.default_masking_strategy()
    print(
        f"{label:16}  players: {type(player).__name__} (n={player.n_players})  "
        f"masking: {type(masking).__name__}"
    )


# Default strategies are defined on the architecture classes; weights are not required.
describe(ResNetArchitecture(model=lambda x: x), "ResNet18")

vit_b16_config = SimpleNamespace(image_size=224, patch_size=16)
describe(
    ViTArchitecture(model=SimpleNamespace(config=vit_b16_config), processor=object()),
    "ViT-B/16",
)
describe(ConvNeXtArchitecture(model=object(), processor=object()), "ConvNeXt-tiny")
describe(DINOv2Architecture(model=object(), processor=object()), "DINOv2-small")
describe(
    CLIPArchitecture(
        model=object(),
        processor=object(),
        text_prompts=["a dog", "a cat"],
    ),
    "CLIP-B/32",
)

ResNet18          players: SuperpixelStrategy (n=10)  masking: MeanColorMasking
ViT-B/16          players: PatchStrategy (n=9)  masking: MaskTokenStrategy
ConvNeXt-tiny     players: SuperpixelStrategy (n=10)  masking: MeanColorMasking
DINOv2-small      players: SuperpixelStrategy (n=10)  masking: MeanColorMasking
CLIP-B/32         players: SuperpixelStrategy (n=10)  masking: MeanColorMasking


## Optional: `ImageExplainer` smoke test

Loads ResNet-18 (~45 MB on first run) and runs a tiny permutation budget. **Skip this section** if you only need the default strategy names above, or use `tutorial_03_benchmarks_and_comparison.ipynb` for full benchmarks.

In [3]:
RUN_SMOKE_TEST = True  # set False to skip ResNet download / forward passes

if not RUN_SMOKE_TEST:
    print("Skipped ImageExplainer smoke test.")
else:
    import torch
    from torchvision import (
        models,
        transforms as T,
    )

    from shapiq import ImageExplainer
    from shapiq.vision import ResNetArchitecture

    # CPU is more stable in Jupyter on some Apple Silicon setups than MPS/CUDA.
    device = torch.device("cpu")
    weights = models.ResNet18_Weights.DEFAULT
    resnet = models.resnet18(weights=weights).eval().to(device)
    preprocess = T.Compose([T.ToTensor(), weights.transforms()])
    pil_image = Image.fromarray(image.astype(np.uint8))
    with torch.no_grad():
        class_id = int(resnet(preprocess(pil_image).unsqueeze(0)).argmax(-1).item())

    def resnet_model(batch_hwc):
        pils = [Image.fromarray(arr.astype(np.uint8)) for arr in batch_hwc]
        tensors = torch.stack([preprocess(im) for im in pils])
        with torch.no_grad():
            return resnet(tensors)[:, class_id].cpu().numpy()

    arch = ResNetArchitecture(model=resnet_model)
    explainer = ImageExplainer(architecture=arch, data=image, index="SV", max_order=1)
    result = explainer.explain_function(image, budget=32)
    values = result.values
    print("values shape:", values.shape)
    print("top-3 players by |value|:", np.argsort(np.abs(values))[-3:][::-1])

values shape: (11,)
top-3 players by |value|: [3 6 8]
